# Antrenare model reciclare (sticla / PET / doza) pe GPU gratuit

Ruleaza celulele pe rand (Shift+Enter sau butonul ▶).

**IMPORTANT — activeaza GPU-ul gratuit:** in meniu sus -> `Runtime` -> `Change runtime type` -> la `Hardware accelerator` alege **GPU** -> Save.

La final descarci fisierul `best.pt` si il pui local in folderul:
`runs/detect/reciclare/weights/best.pt` (langa pet.py). Apoi `python pet.py` il foloseste automat.

## 1. Verifica GPU-ul si instaleaza librariile

In [ ]:
!nvidia-smi
!pip -q install ultralytics roboflow

## 2. Descarca setul de imagini etichetate (Roboflow)

1. Fa-ti cont gratuit pe https://roboflow.com (poti cu Google).
2. Ia-ti cheia API: https://app.roboflow.com/settings/api (copiaza `Private API Key`).
3. Inlocuieste `CHEIA_TA` mai jos cu cheia ta si ruleaza celula.

Setul implicit `updated-recycling-dataset` contine clase precum `PET`, `metal_can`, `glass_bottle` — exact sticla / PET / doza.

**Daca celula da eroare** (versiune gresita / set indisponibil): intra pe orice set de pe https://universe.roboflow.com (cauta `recycling glass plastic can`), apasa **Download Dataset → format YOLOv8 → Show download code** si **inlocuieste** liniile `project = ...` / `dataset = ...` de mai jos cu codul afisat acolo (contine deja workspace, project si versiunea corecta).

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="CHEIA_TA")  # <-- pune cheia ta aici
project = rf.workspace("recyclestuff").project("updated-recycling-dataset")
dataset = project.version(1).download("yolov8")

print("Dataset descarcat in:", dataset.location)
DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml:", DATA_YAML)

In [ ]:
# Afiseaza clasele din set (verifica sa apara sticla/PET/doza)
import yaml
with open(DATA_YAML) as f:
    info = yaml.safe_load(f)
print("Clase in set:", info.get("names"))

## 3. Antreneaza modelul (pe GPU dureaza ~10-20 min)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # model mic, rapid
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name="reciclare",
    patience=15,
)

## 4. Testeaza pe cateva imagini din set (optional)

In [ ]:
from ultralytics import YOLO
import glob
from IPython.display import Image, display

best = "runs/detect/reciclare/weights/best.pt"
m = YOLO(best)
test_imgs = glob.glob(dataset.location + "/valid/images/*.jpg")[:4]
res = m(test_imgs, save=True)
for p in glob.glob("runs/detect/predict*/**/*.jpg", recursive=True)[:4]:
    display(Image(p))

## 5. Descarca modelul antrenat (best.pt)

Dupa descarcare, pune fisierul local in `runs/detect/reciclare/weights/best.pt`
(creeaza folderele daca nu exista), langa `pet.py`. Apoi ruleaza local `python pet.py`.

In [ ]:
from google.colab import files
files.download("runs/detect/reciclare/weights/best.pt")